In [ ]:
#@title 利用モジュール

!pip install biopython

from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/my-modules')
from Bio.PDB import PDBList               #https://qiita.com/Ag_smith/items/94c4b97729b043fae0cb
from Bio.PDB.MMCIF2Dict import MMCIF2Dict #https://biopython.org/docs/latest/api/Bio.html#subpackages
from Bio.PDB.MMCIFParser import FastMMCIFParser

import os
import csv
import shutil
import datetime
import pytz
import re
import requests
from lxml import etree
import pandas as pd                       #https://note.nkmk.me/pandas/
import gzip
from mimetypes import guess_type
import warnings
from itertools import combinations        #https://note.nkmk.me/python-math-factorial-permutations-combinations/
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numba import jit
from decimal import Decimal, ROUND_HALF_UP

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#@title 関数

!pip install biopython
!pip install numba

from Bio.PDB import PDBList  # BioPython から PDBList をインポート

# PDB ファイルをダウンロードする関数
def downloadpdb(pdbid):
    """
    PDB ファイルをダウンロードする関数
    """
    pdb_list = PDBList()  # PDBList クラスを使用
    pdb_list.retrieve_pdb_file(pdbid, pdir="pdb_files/", file_format="pdb")


class UniprotData:
    """
    ユニプロットのXMLデータにアクセスし、情報を取得
    """
    def __init__(self, uniprot_id: int):
        url = f"https://www.uniprot.org/uniprot/{uniprot_id}.xml"
        self.xml = etree.fromstring(requests.get(url).content)
        self.nsmap = self.xml.nsmap
        TF = self.xml.find('./', self.nsmap).text
        if TF != '\n  ':
            raise KeyError(TF)

    def get_pdb_entries(self):
        """PDBエントリを取得"""
        pdb_entries = self.xml.findall(".//{http://uniprot.org/uniprot}dbReference[@type='PDB']", self.nsmap)
        return pdb_entries

    # DSA解析用の閾値
pdb_threshold = 1
chain_threshold = 3

def count_pdb(uniprotid, methods=None):
    """選択した構造決定法(methods)に限定して PDB 数をカウント"""
    # methods が指定されなければ、グローバル定義を使う
    if methods is None:
        methods = METHODS_SELECTED

    unidata = UniprotData(uniprotid)
    pdblist = unidata.pdblist(methods)  # ← methodsを引数で受け取るように修正

    # 除外ID処理
    if negative_pdbid != "":
        negative_list = re.split(r'[,\s]+', negative_pdbid.strip())
        negative_list_upper = [neg.upper() for neg in negative_list]
        pdblist = [item for item in pdblist if item.upper() not in negative_list_upper]

    # PDB数がしきい値以上ならTrue
    return len(pdblist) >= pdb_threshold






    def get_id(self):
        """
        UniProt ID取得
        """
        return [accession.text for accession in self.xml.findall('./entry/accession', self.nsmap)]

    def fasta(self) -> str:
        """
        FASTA 配列の取得
        """
        return self.xml.find('./entry/sequence', self.nsmap).text

    def get_fullname(self):
        """
        fullName取得
        """
        return self.xml.find('./entry/protein/*/fullName', self.nsmap).text

    def get_organism(self):
        """
        organism取得
        """
        return self.xml.find('./entry/organism/name[@type="scientific"]', self.nsmap).text

# クラス UniprotData 内
def getpdbdata(self, method):
    """
    PDBID, method, resolution の取得
    - method: 文字列（"X-ray,EM" など）またはイテラブル（{"X-ray","EM"} など）
    """
    # 受け取った引数を集合化（複数対応）
    if isinstance(method, str):
        methods = {m.strip() for m in re.split(r'[,\s]+', method) if m.strip()}
    else:
        methods = set(method)  # list/tuple/set を想定

    # ひとつも指定が無ければ全種を対象（保険）
    if not methods:
        methods = {"X-ray", "NMR", "EM"}

    pdbid = []
    data = []
    for dbReference in self.xml.findall('./entry/dbReference[@type="PDB"]', self.nsmap):
        x = []
        for propertys in dbReference:
            value = propertys.attrib["value"]
            x.append(value)
            # 既存仕様: NMR は resolution が無いので None を入れる
            if value == 'NMR':
                x.append(None)

        # 旧: x[0] == method  →  新: x[0] in methods
        if x and (x[0] in methods):
            pdbid.append(dbReference.attrib["id"])
            data.append(x)

    self.pdbdata = pd.DataFrame(
        data,
        index=pdbid,
        columns=['method', 'resolution', 'position']
    ).T
    return self.pdbdata

    def pdblist(self, method = ""):
        """
        PDBid取得
        """
        try:
            return self.pdbdata.columns.tolist()
        except AttributeError:
            return self.getpdbdata(method).columns.tolist()

    def position(self, pdbid):
        """
        positionの取得
        """
        positiondata = self.pdbdata.at["position", pdbid].split(", ")
        if len(positiondata) == 1:
            _, posi = positiondata[0].split("=")
            beg, end = posi.split("-")
            beg = int(beg); end = int(end)
        else:
            beg=[];end=[]
            for position in positiondata:
                _, posi = position.split("=")
                align_beg, align_end = posi.split("-")
                beg.append(int(align_beg));end.append(int(align_end))
            beg = min(beg); end = max(end)
        return beg, end


def convert_three(sequence):
    dic = {"A":"ALA", "B":"D|N", "C":"CYS", "D":"ASP", "E":"GLU", "F":"PHE", "G":"GLY", "H":"HIS", "I":"ILE", "K":"LYS", "L":"LEU", "M":"MET", \
           "N":"ASN", "O":"HYP", "P":"PRO", "Q":"GLN", "R":"ARG", "S":"SER", "T":"THR", "U":"SEC", "V":"VAL", "W":"TRP", "X":"any", "Y":"TYR", "Z":"E|Q"}
    return [dic[char] for char in sequence]


pdb_list = PDBList()
def downloadpdb(pdbid):
    """
    Download PDB File
    """
    pdb_list.retrieve_pdb_file(pdbid, pdir="pdb_files/", file_format="mmCif")


def _open(pdbid):
    file = pdbid.lower()+".cif"
    ciffile = "pdb_files/"+file
    if guess_type(file)[1] == "gzip":
        return gzip.open(ciffile, mode='rt')
    else:
        return open(ciffile)


class CifData:
    """
    Cifファイルを解析
    配列情報を取得
    https://mmcif.pdbj.org/docs/pdb_to_pdbx_correspondences.html#DBREF
    https://mmcif.pdbj.org/dictionaries/mmcif_pdbx_v50.dic/Items/index.html
    """
    def __init__(self, pdbid):
        self.pdbid = pdbid
        downloadpdb(self.pdbid)
        with _open(self.pdbid) as handle:
            mmcifdict = MMCIF2Dict(handle)
#        print(mmcifdict)
        self.struct_ref_seq = pd.DataFrame({
            "strand_id": mmcifdict["_struct_ref_seq.pdbx_strand_id"],                         #chain_id
            "accession": [i.upper() for i in mmcifdict["_struct_ref_seq.pdbx_db_accession"]], #uniprot_id
            "seq_align_beg": mmcifdict["_struct_ref_seq.seq_align_beg"],                      #list用
            "seq_align_end": mmcifdict["_struct_ref_seq.seq_align_end"]})
        pdb_strand_id = mmcifdict["_pdbx_poly_seq_scheme.pdb_strand_id"]
        for i, struct_strand_id in enumerate(self.struct_ref_seq["strand_id"]):
            self.struct_ref_seq.at[i, "sort_index"] = pdb_strand_id.index(struct_strand_id)   # sort_index値を"strand_id"でのpdb_strand_idのindex値とする
        self.struct_ref_seq.sort_values("sort_index", inplace=True)
        try:
            self.struct_ref_seq_dif = pd.DataFrame({
                "strand_id": mmcifdict["_struct_ref_seq_dif.pdbx_pdb_strand_id"],
                "seq_num": mmcifdict["_struct_ref_seq_dif.pdbx_auth_seq_num"],
                "db_seq_num": mmcifdict["_struct_ref_seq_dif.pdbx_seq_db_seq_num"],
                "details": [i.lower() for i in mmcifdict["_struct_ref_seq_dif.details"]]})
            self.struct_ref_seq_dif = self.struct_ref_seq_dif[self.struct_ref_seq_dif['details'] != "expression tag"]
            self.struct_ref_seq_dif = self.struct_ref_seq_dif[self.struct_ref_seq_dif['details'] != "linker"]
            self.struct_ref_seq_dif = self.struct_ref_seq_dif[self.struct_ref_seq_dif['details'] != "conflict"]
            self.struct_ref_seq_dif = self.struct_ref_seq_dif[self.struct_ref_seq_dif['details'] != "microgeterogeneity"]
        except KeyError:
            self.struct_ref_seq_dif = pd.DataFrame({"strand_id": [],"seq_num": [],"db_seq_num": []})
        self.chain = []
        self.chainid = []
        self.hetero_info = []
        self.ind = -1
        hetero_pdb_seq_num = ""
        for pdb_mon_id, pdb_seq_num, hetero, chainid in zip(mmcifdict["_pdbx_poly_seq_scheme.pdb_mon_id"], mmcifdict["_pdbx_poly_seq_scheme.pdb_seq_num"], mmcifdict["_pdbx_poly_seq_scheme.hetero"], mmcifdict["_pdbx_poly_seq_scheme.pdb_strand_id"]):
            self.ind += 1
            if hetero == "n":
                hetero_pdb_seq_num = ""
                if pdb_mon_id != "?":
                    self.chain.append(pdb_mon_id +", "+ pdb_seq_num)    # chain は pdb_mon_id, pdb_seq_num の形式で記述（ALA, 1 など）
                    self.chainid.append(chainid)
                else:
                    self.chain.append(None)
                    self.chainid.append(chainid)
            else:
                if pdb_seq_num == hetero_pdb_seq_num:               # _pdbx_poly_seq_scheme.hetero == y かつ 一つ上と同じ_pdbx_poly_seq_scheme.pdb_seq_num の場合、chainにappendしない
                    self.hetero_info.append(self.ind)               # self.chainからChainごとに分割するときの seq_align_beg, seq_align_end の補正のための情報取得
                    continue
                else:
                    if pdb_mon_id != "?":
                        self.chain.append(pdb_mon_id +", "+ pdb_seq_num)
                        self.chainid.append(chainid)
                        hetero_pdb_seq_num = pdb_seq_num
                    else:
                        self.chain.append(None)
                        self.chainid.append(chainid)
        # 重複除去処理の反映
        for j, strandid in enumerate(self.struct_ref_seq["strand_id"]):
            self.struct_ref_seq.at[j, "sort_index"] = self.chainid.index(strandid)   # sort_index値を"strandid"でのchainid.index値に変更する
        atom_coord = pd.DataFrame({
                        "model_num": mmcifdict["_atom_site.pdbx_PDB_model_num"],
                        "asym_id": mmcifdict["_atom_site.auth_asym_id"],
                        "comp_id": mmcifdict["_atom_site.auth_comp_id"],
                        "seq_id": mmcifdict["_atom_site.auth_seq_id"],
#                        "seq_id": mmcifdict["_atom_site.label_seq_id"],
                        "atom_id": mmcifdict["_atom_site.auth_atom_id"],
                        "Cartn_x": mmcifdict["_atom_site.Cartn_x"],
                        "Cartn_y": mmcifdict["_atom_site.Cartn_y"],
                        "Cartn_z": mmcifdict["_atom_site.Cartn_z"],
                        "alt_id": mmcifdict["_atom_site.label_alt_id"],
                        "group_PDB": mmcifdict["_atom_site.group_PDB"],
                        "ins_code": mmcifdict["_atom_site.pdbx_PDB_ins_code"]
                        })
        atom_coord["asym_id"] = atom_coord["asym_id"].astype(str)                # asym_id が数字だった場合への対応 --> 後にCSVファイルへ出力して再度Pandas DataFrameにしているので、そのときにもう一度データ型指定する
#        atom_coord = atom_coord[(atom_coord['group_PDB'] == 'ATOM') & (atom_coord['alt_id'] != 'A')].drop(columns=['alt_id', 'group_PDB'])  #※※※ alt_idの指定で必要な情報が削除されてしまう場合がある
        # alt_idを検索してユニーク化
        # 元のインデックスを保持するための列を追加
        atom_coord['original_index'] = atom_coord.index
        # 'alt_id' 列が '.' である行をそのままにする
        alt_id_dot = atom_coord[atom_coord['alt_id'].str.contains('\\.')]
        # 'alt_id' 列が '.' でない行をフィルタリング
        alt_id_not_dot = atom_coord[~atom_coord['alt_id'].str.contains('\\.')]
        # 同じ 'seq_id' の数字を持つ行について 'atom_id' 列が一意になるようにフィルタリング
        alt_id_not_dot_unique = alt_id_not_dot.drop_duplicates(subset=['seq_id', 'atom_id'])
        # 'alt_id' 列が '.' である行と '.' でないがユニークな 'atom_id' を持つ行を結合
        atom_coord = pd.concat([alt_id_dot, alt_id_not_dot_unique])
        # 元のインデックスでソート
        atom_coord = atom_coord.sort_values('original_index')
        # 'original_index' 列を削除
        atom_coord = atom_coord.drop(columns=['original_index'])
        atom_coord = atom_coord[(atom_coord['group_PDB'] == 'ATOM')].drop(columns=['alt_id', 'group_PDB'])
        if not(os.path.exists('atom_coord/')):
            os.makedirs('atom_coord/')
        atom_coord.to_csv(f'atom_coord/{self.pdbid}.csv', index=False)

    def mutationjudge(self, uniprotids, pdbid):
        m_pd = self.struct_ref_seq[["strand_id", "accession"]]                #PDBに含まれる全ChainのUniProtID情報を取得
        unim_pd = m_pd[m_pd["accession"].isin(uniprotids)]                    #入力したUniProtIDと一致したChainのみ抽出#XMLに記載のUniProtIDと一致したものの抽出
        if unim_pd["accession"].count() == 0:                                 ### 入力UniProtIDと一致したChainはゼロのとき
            if vervose: print(uniprotids, 'not matched. UniProt ID(s) in this PDB is listed below')
            exunim_pd = m_pd[m_pd["accession"] != (uniprotids and pdbid)]     ### 入力UniProtIDとPDBIDに一致したChainは除外
            if vervose: print(exunim_pd["accession"].unique())                ### 入力UniProtIDと一致していないUniProtID ==> ※※※この配列が入力UniProtIDの配列と一致（identity>90%?）するなら、このUniProtIDのChainも処理をつづける
            return "UniProt ID mismatch"
#            return "normal"                                                   ### ※※一時的にUniProtIDが一致していないものもnormalとする？※※
        else:
            if unim_pd.duplicated().sum() != 0:                               #AAAAAA-BBBBBB-AAAAAAのように同じChainに複数回同じUniProtIDが登場する場合
                return "chimera"                                              #chimeraと判定
#            print(unim_pd["strand_id"])
            m_id = list(unim_pd["strand_id"])                                 #入力したUniProtIDと一致したChain番号
#            print(m_id)
            mdif_pd = self.struct_ref_seq_dif[self.struct_ref_seq_dif["strand_id"].isin(m_id)]    #headerのmutation情報を取得
            if len(mdif_pd) == 0:                                             #変異情報がなければnormalと判定
                return "normal"
            else:
                mdif_pd_details = mdif_pd["details"].unique()
                np.sort(mdif_pd_details)
                if "engineered mutation" in mdif_pd_details:
                    if vervose: print("engineered mutation")
                    return "substitution"
                elif "microheterogeneity" in mdif_pd_details:
                    if vervose: print("microheterogeneity")
                    return "normal"
            s_list = list(m_pd["strand_id"])                                  #PDBに含まれる全Chain（UniProtごとに別とみなす）のChainID情報を取得
            if len(s_list) != len(set(s_list)):                               #PDBに含まれる全Chain数と重複除去したChain数が異なる場合chimeraと判定
                return "chimera"                                              #duplicated(subset='strand_id')でも判定可能
            for i in m_id:
                strand_mdif_pd = mdif_pd[mdif_pd["strand_id"] == i]
                seq_num_list = list(strand_mdif_pd["seq_num"])
                db_seq_num_list = list(strand_mdif_pd["db_seq_num"])
#                print(len(seq_num_list))
#                print(len(db_seq_num_list))
                if len(seq_num_list) != len(set(seq_num_list)):
                    return "delins"
                elif len(db_seq_num_list) != len(set(db_seq_num_list)):
                    return "delins"
            return "substitution"


    def getsequence(self, uniprotids):
        firstLoop = True
        struct = self.struct_ref_seq[self.struct_ref_seq['accession'].isin(uniprotids)].drop_duplicates(subset=["strand_id"])
#        print(struct)
        for row in struct.itertuples():
            if row.accession in uniprotids:
                sort_index = int(row.sort_index)
                align_beg = sort_index + int(row.seq_align_beg)-1
                align_end = sort_index + int(row.seq_align_end)
                chain = self.chain[align_beg:align_end]                         # pdb_mon_id +", "+ pdb_seq_num の形式
                mutat_info = self.struct_ref_seq_dif[self.struct_ref_seq_dif["strand_id"] == row.strand_id].drop(columns='strand_id')
                if len(mutat_info) != 0:
                    #deletionの処理
                    deletion = mutat_info[(mutat_info["seq_num"] == '?')].index
                    if len(deletion) != 0:
                        mutat_info.drop(deletion, inplace=True)
                        chain_num = pd.Series(chain).map(lambda x: int(x.split(', ')[1]) if isinstance(x, str) else x).diff()
                        deletion = chain_num[(chain_num != 1)].dropna()
                        for index, i in zip(deletion.index, deletion):
                            chain[index:index] = [None]*int(i)
                    #insertion
                    insertion = mutat_info[(mutat_info["db_seq_num"] == '?')]["seq_num"]
                    if len(insertion) != 0:
                        mutat_info.drop(insertion.index, inplace=True)
                        insertion = insertion.values.tolist()
                        print(insertion)
                        for i in chain:
                            if isinstance(i, str):
                                for n in insertion:
                                    if n == i.split(', ')[1]:
                                        insertion.remove(n)
                                        chain.remove(i)
                    #delins
                    #del
                    dup_mutat = mutat_info[mutat_info.duplicated(subset=["seq_num"], keep=False)]
                    if len(dup_mutat) != 0:
                        mutat_info.drop(dup_mutat.index, inplace=True)
                        for i in dup_mutat['seq_num'].drop_duplicates():
                            chain_num = pd.Series(chain).map(lambda x: int(x.split(', ')[1]) if isinstance(x, str) else x)
                            num = len(dup_mutat[dup_mutat['seq_num'] == i])-1
                            index = chain_num[chain_num == int(i)].index[0] +1
                            chain[index:index] = [None]*num
                    #ins
                    dup_mutat = mutat_info[mutat_info.duplicated(subset=["db_seq_num"], keep=False)]
                    if len(dup_mutat) != 0:
                        mutat_info.drop(dup_mutat.index, inplace=True)
                        insertion = []
                        for i in dup_mutat['db_seq_num'].drop_duplicates():
                            insertion += dup_mutat[dup_mutat['db_seq_num'] == i]["seq_num"].reset_index(drop=True).drop([0]).values.tolist()
                        m = 0
                        for i in range(len(chain)):
                            i = chain[i+m]
                            if isinstance(i, str):
                                for n in insertion:
                                    if n == i.split(', ')[1]:
                                        insertion.remove(n)
                                        chain.remove(i)
                                        m -=1

                if firstLoop:
                    firstLoop = False
                    sequence = pd.DataFrame(chain, columns = [self.pdbid +' '+row.strand_id])
                else:
                    strand = pd.Series(chain, name=self.pdbid +' '+row.strand_id)
                    sequence = pd.concat([sequence, strand], axis=1)

        if firstLoop:
            sequence = pd.DataFrame()
#            print(sequence.columns.values)
#        print("getseq_sequence")
#        print(sequence)
        return sequence


def trim_sequence(sequencedata, seq_ratio = 80):
    """
    座標データが存在しているアミノ酸残基の数がuniprotの配列長に対する割合（seq_ratio）以下であれば、削除する
    欠損値がある行を削除
    """
    sequencedata.dropna(subset=sequencedata.columns[0], inplace=True)
    seqlen = len(sequencedata)
    delchain = [chain for chain, item in sequencedata.items() if 100 - (item.isnull().sum()/seqlen*100) < seq_ratio]
    seqdata = sequencedata.drop(columns = delchain)
    seqdata.dropna(inplace=True)
    return seqdata


def _diff(uniprotid, df1, df2, shift = 0):
    diff = pd.concat([df1, df2.shift(shift)], axis=1)
    diff.dropna(inplace=True)     # デフォルトで how='any' --> 欠損値が一つでもある行を削除、inplace=Trueなので元のオブジェクト自体を変更
    diff.drop_duplicates(subset=uniprotid, ignore_index=True, inplace=True) # UniProtIDsカラムで重複削除
#    print(diff)
    return (diff.iloc[:, 0] == diff.iloc[:, 1]).sum()

def trim2_sequence(sequencedata, seq_ratio = 80):
    """
    seq_id重複の場合、最初だけを残して、残りは削除する
    """
#    seq = sequencedata.map(lambda x: x.split(', ')[1] if isinstance(x, str) else x)    # sequencedateは pdb_mon_id, pdb_seq_num の情報、ここで pdb_seq_num だけにする
#    seq = sequencedata.map(lambda x: int(x.split(', ')[1]) if isinstance(x, int) else x)
    seq = sequencedata.iloc[:, 1:].map(lambda x: int(x.split(', ')[1]))   # sequencedateは pdb_mon_id, pdb_seq_num の情報、ここで pdb_seq_num だけにする
#    print(seq[40:70])
    # 重複する行のインデックスを格納するリストを初期化
    duplicate_indices = set()
    # 各カラムを走査して重複行のインデックスを収集
    for column in seq.columns:
        duplicates = seq[column].duplicated(keep='first')
        duplicate_indices.update(seq[duplicates].index)
    # 重複する行のインデックスをリストとして取得
    duplicate_indices = sorted(list(duplicate_indices))
    # 重複する行を削除した新しいDataFrameを作成
    trim2_seq = sequencedata.drop(index=duplicate_indices)
    # 重複した行のインデックスを表示
#    print(f"Duplicate indices: {duplicate_indices}")
    # 重複行が削除されたDataFrameを表示
#    print(trim2_seq[40:70])
#    return sequencedata
    return trim2_seq


def sort_sequence(uniprotid, sequencedata, seq_ratio):
    """
    チェック機構
    """
    seq = sequencedata.map(lambda x: x.split(', ')[0] if isinstance(x, str) else x)    # sequencedateは pdb_mon_id, pdb_seq_num の情報、ここで pdb_mon_id だけにする
    trimdata = trim_sequence(seq, seq_ratio)                                          # pdb_mon_id（アミノ酸残基情報）のみでtrimming
#    uniprot_id = trimdata.columns[0]
    trimdata.drop_duplicates(subset=uniprotid, ignore_index=True, inplace=True) # 重複削除（アミノ酸一つずつにする）
    trimdata.reset_index(inplace=True, drop=True)
    trimdata = trimdata.T         # 縦横変換（行：アミノ酸、列：Chain --> 行：Chain、列：アミノ酸）
#    print(trimdata)
    columns = trimdata.columns    # 列名（index）の取得
    IDs = []                      # 他とずれているIDを取得
    for col in columns:           # uniprotidの文字列で検索かけてbool判定
        diff = trimdata[trimdata[col] != trimdata.at[uniprotid, col]].index   # uniprotidのアミノ酸と異なるChainのリスト
#        print(col, diff.values)
        if len(diff) !=0:
            IDs.extend(diff)
            trimdata.drop(diff, inplace=True)  # falseの行の行名をIDsに追加しその行を消す ※※※不要？
#            print(trimdata)
    uniseq = seq[uniprotid]       # UniProt配列
#    print(IDs)
    for ID in IDs:                # ずれている配列の処理
        difseq = seq[ID]
        unique = _diff(uniprotid, uniseq, difseq) # uniseqとdifseqの一致度を確認（アミノ酸20種類を出現順に並べたときのマッチしている数）
        if unique > 10:           # 11個以上一致しているならば、そのまま使用する（変異だとみなして許容する）
            continue
        num = 1; unique = 0
        while unique < 10 and num < 100:          # 配列調整 プラスマイナス100
            unique = _diff(uniprotid, uniseq, difseq, num)
            num = (-num)+1 if num < 0 else -num
        # uniと一致しているアミノ酸が11個以上になったらそれをsequencedataのその列と置換する(元の列消しshiftした列代入)
        # 10個以下であれば、seqencedataから削除して使用しない
        if unique > 10:
            diff = sequencedata[ID].shift((-num)+1 if num > 0 else -num)
            loc = sequencedata.columns.get_loc(ID)
            sequencedata.drop(ID, axis=1, inplace=True)
            sequencedata.insert(loc, ID, diff)
        else:
            print(ID, "is not used due to sequence alignment failure")
            loc = sequencedata.columns.get_loc(ID)
            sequencedata.drop(ID, axis=1, inplace=True)
#    print("seqdata", sequencedata)
#    print("trim_seqdata", trim_sequence(sequencedata, seq_ratio))
#    return trim_sequence(sequencedata, seq_ratio)
    sorted_seqdata = trim_sequence(sequencedata, seq_ratio)
    uniq_sorted_seqdata = trim2_sequence(sorted_seqdata)
#    print(uniq_sorted_seqdata[40:70])
    return uniq_sorted_seqdata

def getcoord(trimsequence):
    atomcoord = pd.DataFrame(trimsequence.iloc[:, 0])         # Uniprot配列
    atomindex = atomcoord.index.tolist()                      # atomcoordのindexをリストで取得
    trimseq = trimsequence.iloc[:, 1:].map(lambda x: int(x.split(', ')[1]))   # seqのみに
    columns = trimseq.columns.tolist()                        # PDB ID リスト
#    print(columns)
    pdbids = {}
    for col in columns:
        pdbid, strand_id = col.split(' ')
        pdbids.setdefault(pdbid, []).append(strand_id)
    for pdbid, chain_id in pdbids.items():
        struct = pd.read_csv(f'atom_coord/{pdbid}.csv')
        struct["asym_id"] = struct["asym_id"].astype(str)                       # asym_id列が数字の場合（chain idが数字となっている場合）に対応するため、データ型をstrに変換
        struct = struct[struct["atom_id"] == "CA"]
        struct.drop(columns=['model_num', 'atom_id'], inplace=True)   # model_num と atom_id カラム を削除
        for chain in chain_id:
            seq_num = trimseq[pdbid +' '+chain]                                 # カラムから seq_num 要素を Pandas.Series として取得
            seq_num.index = seq_num.tolist()                                    # seq_numの要素をindexへ反映
#            print("seq_num", len(seq_num))
            chaindata = struct[struct["asym_id"] == chain]                      # asym_idがchain変数と同一である行をstructから取得し、chaindata DataFrameとする *** asym_idが数字の場合に対応させる必要がある asym_id列のデータ型をstrにする（上で対応）
            chaindata.index = chaindata["seq_id"].tolist()                      # seq_id の情報を index へ反映 ※※ seq_idが重複している場合(_pdbx_poly_seq_scheme.pdb_ins_codeがある場合など) はこの後でエラー：getsequence あるいは sort_sequence のところで何とかできるかも
#            print("chaindata", len(chaindata))
# 座標取得のため、再度structからデータを抽出しているので、seq_id が重複している場合は、その行を削除して対応する
            # 'seq_id'カラムの重複を確認
            if chaindata['seq_id'].duplicated().any():
#                print("'seq_id' duplication was detected")
                # 'seq_id'ごとの重複数をカウント
                seq_id_duplicate_counts = chaindata['seq_id'].value_counts().reset_index()
                # カラム名を設定
                seq_id_duplicate_counts.columns = ['seq_id', 'count']
                # countが2以上のものだけフィルタリング
                seq_id_duplicates_filtered = seq_id_duplicate_counts[seq_id_duplicate_counts['count'] >= 2]
                # seq_idを昇順でソート
                seq_id_duplicates_sorted = seq_id_duplicates_filtered.sort_values(by='seq_id', ascending=True)
                # インデックスを表示せずに結果を表示
#                print(seq_id_duplicates_sorted.to_string(index=False))
                chaindata = chaindata.drop_duplicates(subset='seq_id', keep='first')
#                print(len(chaindata))
#            else:
#                print(len(chaindata))
#            print(len(chaindata))
#            print(chaindata.head(10))
#            print(chaindata[40:60])
#            print(chaindata.tail(10))
#            print(chaindata["seq_id"].tolist())
#            print(chaindata["seq_id"].drop_duplicates().tolist())
            coord = chaindata[['comp_id', 'Cartn_x', 'Cartn_y', 'Cartn_z']]
#            print(coord[40:70])
            coord = chaindata[['comp_id', 'Cartn_x', 'Cartn_y', 'Cartn_z']].filter(items=seq_num.tolist(), axis=0)    # tolist()でpd.Seriesをリストに変換
#            print("filtered.coord")
#            print(coord.tail(10))
#            print(len(coord))
            coord = pd.concat([seq_num, coord], axis=1)                         # pdb_id chain, comp_id, x, y, zの情報
            coord.drop(columns=pdbid +' '+chain, inplace=True)                  # pdb_id chain のseq_numカラムを削除
            coord.rename(columns={'comp_id': pdbid +' '+chain}, inplace=True)   # comp_id（アミノ酸残基名）カラム名を pdb_id chainへリネーム
            coord.index = atomindex
            atomcoord = pd.concat([atomcoord, coord], axis=1)
    atomcoord.dropna(inplace=True)
    return atomcoord

from numba import jit

@jit(nopython=True)
def calculat(atom1, atom2):
    ## 11桁まで正確
    xyz = atom1 - atom2
    xyz = np.rint(xyz*1000)
    dis = np.sqrt(np.sum(xyz**2))     # ***
    return dis/1000


def getdistance(atomcoord):
    id = atomcoord.iloc[:, 0].name
    distance = pd.DataFrame({id : [a+", "+b for a, b in combinations(map(str, atomcoord.index), 2)],
                             "residue pair": [resi0+", "+resi1 for resi0, resi1 in combinations(atomcoord[id], 2)]})  # 残基組み合わせ
    cols = atomcoord.iloc[:, 1::4].columns.tolist()           # カラム名を三つ飛ばし-->PDB ID Chain IDのみ抽出できる
    combination = [i for i in combinations(range(len(atomcoord)), 2)]
    for i, col in enumerate(cols):
#        print(i, col)             # 0 1BBS A など
        i = (i*4)+2
        atoms = atomcoord.iloc[:, i:i+3].to_numpy()         # NumPy ndarray
        distance[col] = [calculat(atoms[n1], atoms[n2]) for n1, n2 in combination]
#    print(distance)
    return distance

def getdistance2(atomcoord):
    id = atomcoord.iloc[:, 0].name                            # UniProt ID(s)
    cols = atomcoord.iloc[:, 1::4].columns.tolist()           # カラム名を三つ飛ばし-->PDB ID Chain IDのみ抽出できる
    num_cols = len(cols) + 2
    num_rows = (len(atomcoord) * (len(atomcoord) - 1)) // 2
    distance = pd.DataFrame({
        #id : [a+", "+b for a, b in combinations(map(str, atomcoord.index), 2)],
        id : [str(int(a)+1)+", "+str(int(b)+1) for a, b in combinations(map(str, atomcoord.index), 2)],
        "residue pair": [resi0+", "+resi1 for resi0, resi1 in combinations(atomcoord[id], 2)],
        **{col: np.nan for col in cols}
    })
#    print(distance)
    combination = list(combinations(range(len(atomcoord)), 2))
    for i, col in enumerate(cols):
#        print(i, col)             # 0 1BBS A など
        i = (i*4)+2
        atoms = atomcoord.iloc[:, i:i+3].to_numpy()
        distance[col] = [calculat(atoms[n1], atoms[n2]) for n1, n2 in combination]
#    print(distance)
    return distance

def getscore(distance, ddof=0):
    """
    ddof=1: 標本標準偏差, ddof=0: 母数標準偏差
    """
    dis = distance.iloc[:, 2:]
    means = dis.mean(axis='columns')
    stds = dis.std(axis='columns',ddof=0)
    stds = stds.map(lambda x: 0.0001 if x==0 else x)
    column0 = distance.columns[0]
    return pd.DataFrame({column0: distance[column0],
                         "residue pair": distance["residue pair"],
                                     "distance mean" : means,
                                      "distance std" : stds,
                                      "score" : means/stds})

def getscore_cis(distance, ddof=0):
    """
    ddof=1: 標本標準偏差, ddof=0: 母数標準偏差
    """
    dis = distance.iloc[:, 2:]
    means = dis.mean(axis='columns')
    stds = dis.std(axis='columns',ddof=0)
    stds = stds.map(lambda x: 0.0001 if x==0 else x)
    column0 = distance.columns[0]
    return pd.DataFrame({column0: distance[column0],
                         "residue pair": distance["residue pair"],
                                     "distance mean" : means,
                                      "distance std" : stds,
                                      "score" : means/stds})

from decimal import Decimal, ROUND_HALF_UP
def generate_log_content(pdbdata, len_sequence, trimsequence, score, cis_info):
    cis_dist_mean, cis_dist_std, cis_score_mean, cis_num, mix = cis_info[0][0], cis_info[0][1], cis_info[0][2], cis_info[0][3], cis_info[0][4]
    cols = trimsequence.columns.values[1:]
    pdbids = [i.split(' ')[0] for i in cols]
    print(pdbids)
    """
    分解能平均値の計算にchainの重複を許可
    """
    reso_list = []
    for pdbid in pdbids:
        reso = pdbdata.at['resolution', pdbid]
        reso = ''.join(char for char in reso if char.isdigit() or char == '.')
        reso_list.append(float(reso))
    reso_ave = np.mean(reso_list)
    reso_ave = Decimal(str(np.mean(reso_list))).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
#    print(reso_ave)
#    print(round(reso_ave, 2))
    """
    分解能平均値の計算にchainの重複を認めない
    """
    seted = sorted(set(pdbids), key=pdbids.index)
    resolution_sum = 0
    resolution = pdbdata[seted].loc["resolution"]
    for i, str_resolution in enumerate(resolution, 1):
        resolution_sum += float(str_resolution.split(' ')[0])
    """
    出力
    """
    return pd.DataFrame({'Entries': [len(seted)],
                        'Chains': [len(pdbids)],
                        'Length': [len(trimsequence)],
                        'Length(%)': [round((len(trimsequence)*100/len_sequence), 1)],
#                        'Resolution': [resolution_sum/i],
#                        'Resolution': [round(reso_ave, 2)],
                        'Resolution': [reso_ave],
                        'UMF': [round((score["distance mean"]/score["distance std"]).mean(), 1)],
                        'cis/Length(%)': [round((cis_num*100/len(trimsequence)), 2)],
                        'mean_cisDist': [round(cis_dist_mean, 2)],
                        'std_cisDist': [round(cis_dist_std, 2)],
                        'mean_cisScore': [round(cis_score_mean, 2)],
                        'cis': [cis_num],
                        'mix': [mix]
                         })


#def export_to_csv(filepath, uniprot_id, seq_ratio, outputdataname ,outputdata):
#    filePath = f"{filepath}{uniprot_id}_{str(seq_ratio)}_{outputdataname}.csv"
#    outputdata.to_csv(filePath, index=False)


"""
ヒートマップの作成
"""
def generate_heatmap(score):
    n0, n1 = score.iloc[-1, 0].split(', ')                               # アミノ酸配列の最終組み合わせを取得（UniProt配列の方がいい？）
    df1 = pd.DataFrame(np.zeros((int(n1), int(n1))))                     # 最大サイズですべての要素がゼロのDataFrame作成
    df1[:] = np.nan                                                      # ゼロをNaNに置換

    pdb_threshold = 1
chain_threshold = 3

def count_pdb(uniprotid, methods=None):
    """選択した構造決定法(methods)に限定して PDB 数をカウント"""
    if methods is None:
        methods = METHODS_SELECTED  # デフォルトは全体設定を使用

    unidata = UniprotData(uniprotid)
    pdblist = unidata.pdblist(methods)  # ← ここが変更ポイント

    if negative_pdbid != "":
        negative_list = re.split(r'[,\s]+', negative_pdbid.strip())
        negative_list_upper = [neg.upper() for neg in negative_list]
        pdblist = [item for item in pdblist if item.upper() not in negative_list_upper]

    return len(pdblist) >= pdb_threshold


def prep(uniprotid, methods=None):
    if methods is None:
        methods = METHODS_SELECTED
    unidata = UniprotData(uniprotid)
    uniprotids = unidata.get_id()
    id = str(uniprotids)
    fasta = unidata.fasta()
    sequence = convert_three(fasta)
    seqdata = pd.DataFrame(sequence, columns=[id])
    len_seqdata = len(seqdata)
    pdblist = unidata.pdblist(methods)

    if negative_pdbid != "":
        negative_list = re.split(r'[,\s]+', negative_pdbid.strip())
        negative_list_upper = [neg.upper() for neg in negative_list]
        pdblist = [item for item in pdblist if item.upper() not in negative_list_upper]

    if vervose: print("  Processing " + str(len(pdblist)) + " PDB entries ...")

    nor_pdblist = []
    sub_pdblist = []
    chi_pdblist = []
    din_pdblist = []

    for n, pdbid in enumerate(pdblist):
        cifdata = CifData(pdbid)
        mut_judge = cifdata.mutationjudge(uniprotids, pdbid)
        if vervose: print(" (" + str(n+1) + "/" + str(len(pdblist)) + ") judge:", pdbid, mut_judge)

        if mut_judge == 'normal':
            nor_pdblist.append(pdbid)
        elif mut_judge == 'substitution':
            sub_pdblist.append(pdbid)
        elif mut_judge == 'chimera':
            chi_pdblist.append(pdbid)
        elif mut_judge == 'delins':
            din_pdblist.append(pdbid)
        else:
            continue

        beg, end = unidata.position(pdbid)
        df_beg = pd.DataFrame(index=list(range(beg-1)))
        df_end = pd.DataFrame(index=list(range(len_seqdata - end)))
        seq = cifdata.getsequence(uniprotids)
        seq = pd.concat([df_beg, seq, df_end])
        seq.reset_index(inplace=True, drop=True)
        seqdata = pd.concat([seqdata, seq], axis=1)

    all_pdblist = [nor_pdblist, sub_pdblist, chi_pdblist, din_pdblist]

    if vervose:
        print(" Data Preparation Finished: " + str(len(nor_pdblist) + len(sub_pdblist) + len(chi_pdblist) + len(din_pdblist)) + "/" + str(len(pdblist)) + " PDB entries, " + str(len(seqdata.columns)-1) + " chains as " + uniprotid)
        print(" (Normal PDB: " + str(len(nor_pdblist)) + ", Substitution PDB: " + str(len(sub_pdblist)) + ", Chimera PDB: " + str(len(chi_pdblist)) + ", DelIns PDB: " + str(len(din_pdblist)) + ")")

    return seqdata, all_pdblist

    def Q(x, df):
        #x00, x01 = x[0].split(', ')
        #df.iat[int(x00)-1,int(x01)-1] = x[4]                            # scoreデータフレームの要素[4]はscore
        x00, x01 = x[0].split(', ')  # 文字列として扱う
        df.loc[int(x00) - 1, int(x01) - 1] = x[4]  # locでアクセス、整数に変換してインデックスを1減らす

    def P(x, df):
        x00, x01 = x[0].split(', ')
        x[3] = 0.0005 if x[3] == 0 else x[3]
        df.iat[int(x00)-1,int(x01)-1] = x[3] / x[2]

    score.apply(Q, df = df1, axis = 1)
    return(df1)

    score.apply(Q, df = df1, axis = 1)
    return(df1)

# ↓↓↓ ここに追加 ↓↓↓

def run_DSA(uniprotid, seqdata, export, seqtype, methods=None):
    if methods is None:
        methods = METHODS_SELECTED

    unidata = UniprotData(uniprotid)
    uniprotids = unidata.get_id()
    str_ids = str(uniprotids)
    fasta = unidata.fasta()
    sequence = convert_three(fasta)
    pdblist = unidata.pdblist(methods)

    negative_pdbid = ""  # 必要なら外側のUI値を使ってください

    if negative_pdbid != "":
        negative_list = re.split(r'[,\s]+', negative_pdbid.strip())
        negative_list_upper = [neg.upper() for neg in negative_list]
        pdblist = [item for item in pdblist if item.upper() not in negative_list_upper]

    if seqdata is None or len(seqdata) == 0:
        print(f"Error: seqdata is empty for {uniprotid}")
        return None, "", None

    trimsequence = sort_sequence(str_ids, seqdata, seq_ratio)
    if trimsequence is None or len(trimsequence) == 0:
        print(f"Error: trimsequence is empty for {uniprotid}")
        return None, "", None

    trimsequence.to_csv(os.path.join(dirpath, f"trimsequence_{uniprotid}.csv"), index=False)
    trimseqcol = trimsequence.columns.values[1:]

    if len(trimseqcol) <= chain_threshold - 1:
        print(f"Error: Not enough chains for {uniprotid}")
        return None, "", None

    atomcoord = getcoord(trimsequence)
    if atomcoord is None or len(atomcoord) == 0:
        print(f"Error: atomcoord is empty for {uniprotid}")
        return None, "", None

    distance = getdistance2(atomcoord)
    if distance is None or len(distance) == 0:
        print(f"Error: distance is empty for {uniprotid}")
        return None, "", None

    score = getscore(distance, 0)

    # 残基ペアCSV（必要なければそのままでOK）
    residue_pairs = list(combinations(atomcoord.index, 2))
    residue_num1_list = [pair[0] + 1 for pair in residue_pairs]
    residue_num2_list = [pair[1] + 1 for pair in residue_pairs]
    residue_num_df = pd.DataFrame({'residue_num1': residue_num1_list, 'residue_num2': residue_num2_list})
    distance_cols = distance.columns[2:]
    distance_data_df = distance[distance_cols].copy()
    merged_df = pd.concat([residue_num_df, distance_data_df], axis=1)
    merged_df.to_csv(os.path.join(dirpath, f"distance_{uniprotid}.csv"), index=False, header=False)

    # cis解析
    cis_index = []
    for col in distance.columns.values.tolist()[2:]:
        tmp = distance.query(f'`{col}`<=@cis_threshold').index.to_list()
        cis_index.extend(tmp)

    if not cis_index:
        cis_info = [[0, 0, 0, 0, 0]]
        cis_dist = pd.DataFrame()
    else:
        cis_index = sorted(set(cis_index))
        cis_dist = distance.iloc[cis_index, :]
        cis_cnt = cis_dist.iloc[:, 2:].apply(lambda row: (row <= cis_threshold).sum(), axis=1)
        trans_cnt = cis_dist.iloc[:, 2:].apply(lambda row: (row > cis_threshold).sum(), axis=1)
        # 代表値（平均など）を作るならここで計算して cis_info に入れてもOK

        # 例: 距離とスコアの平均（適宜置き換え）
        cis_dist_mean = cis_dist.iloc[:, 2:].mean(axis=None)
        cis_dist_std  = cis_dist.iloc[:, 2:].stack().std(ddof=0)
        cis_score_mean = (cis_dist.iloc[:, 2:].mean(axis=1) / (cis_dist.iloc[:, 2:].std(axis=1).replace(0, 0.0001))).mean()
        cis_num = len(cis_dist)
        mix = 0
        cis_info = [[cis_dist_mean, cis_dist_std, cis_score_mean, cis_num, mix]]

    # ここが重要：summary DataFrame を直接作る
    summary_df = generate_log_content(unidata.pdbdata, len(sequence), trimsequence, score, cis_info)

    # 従来の log 互換の2行テキストも作っておく（既存コードが使っている場合に備え）
    header_line = " ".join(summary_df.columns.astype(str).tolist())
    value_line  = " ".join(str(v) for v in summary_df.iloc[0].tolist())
    log_text = header_line + "\n" + value_line

    return score, log_text, summary_df



In [3]:
#@title DSAコード

#@title 関数

!pip install biopython
!pip install numba

# ... ドキュメント8の全内容 ...
# （UniprotData, CifData, count_pdb, prep, run_DSAなどすべての関数）

pdb_threshold = 1
chain_threshold = 3             # 標準偏差を出すため、最低でも3つのChainが必要（DSAでなければ1でも可）
#cis_threshold = 3.3               # Cisぺプチド結合のCA-CA間距離

def count_pdb(uniprotid):
    unidata = UniprotData(uniprotid)
    pdblist = unidata.pdblist(method)                         # 入力UniProtIDに関しmethodで決定されたPDBリスト
    if negative_pdbid != "":
        # negative_pdbidに含まれる要素をpdblistから削除
        # negative_pdbidに含まれるPDBをpdblistから削除（pdblistにない場合にも対応）
        # negative_pdbidをスペースまたはカンマで分割してリストに変換
        negative_list = re.split(r'[,\s]+', negative_pdbid.strip())
        # negative_listをすべて大文字に変換
        negative_list_upper = [neg.upper() for neg in negative_list]
        # pdblistの要素を大文字にして比較し、negative_listに含まれる要素を削除
        pdblist = [item for item in pdblist if item.upper() not in negative_list_upper]
#        for removepdbid in negative_pdbid.split():
#            pdblist.remove(removepdbid.upper())
    if len(pdblist) >= pdb_threshold:
        return True
    else:
        return False

def prep(uniprotid):
    unidata = UniprotData(uniprotid)
    uniprotids = unidata.get_id()                             # あるUniProt IDのXMLに含まれる（複数の）UniProt ID
    id = str(uniprotids)                                      # あるUniProt IDのXMLに含まれる（複数の）UniProt ID
    fasta = unidata.fasta()
    sequence = convert_three(fasta)
    seqdata = pd.DataFrame(sequence, columns = [id])          # UniProt配列をseqdata(DataFrame)のid=str(uniprotids)ラベルした列に
    len_seqdata = len(seqdata)                                # UniProt配列長
    pdblist = unidata.pdblist(method)                         # 入力UniProtIDに関しmethodで決定されたPDBリスト
    if negative_pdbid != "":
        # negative_pdbidに含まれる要素をpdblistから削除
        # negative_pdbidに含まれるPDBをpdblistから削除（pdblistにない場合にも対応）
        # negative_pdbidをスペースまたはカンマで分割してリストに変換
        negative_list = re.split(r'[,\s]+', negative_pdbid.strip())
        # negative_listをすべて大文字に変換
        negative_list_upper = [neg.upper() for neg in negative_list]
        # pdblistの要素を大文字にして比較し、negative_listに含まれる要素を削除
        pdblist = [item for item in pdblist if item.upper() not in negative_list_upper]
#        for removepdbid in negative_pdbid.split():
#            pdblist.remove(removepdbid.upper())
    if vervose: print("  Processing " + str(len(pdblist)) + " PDB entries ...")
    nor_pdblist = []
    sub_pdblist = []
    chi_pdblist = []
    din_pdblist = []
    all_pdblist = []
    for n, pdbid in enumerate(pdblist):
        cifdata = CifData(pdbid)
        mut_judge = cifdata.mutationjudge(uniprotids, pdbid)
        if vervose: print(" (" + str(n+1) + "/" + str(len(pdblist)) + ") judge:", pdbid, mut_judge)
        if mut_judge == 'normal':
            nor_pdblist.append(pdbid)
        elif mut_judge == 'substitution':
            sub_pdblist.append(pdbid)
        elif mut_judge == 'chimera':
            chi_pdblist.append(pdbid)
        elif mut_judge == 'delins':
            din_pdblist.append(pdbid)
        else:
            continue
        beg, end = unidata.position(pdbid)                    # UniProt XMLに記載のPositions
        df_beg = pd.DataFrame(index=list(range(beg-1)))       # UniProt配列と長さを合わせるため
        df_end = pd.DataFrame(index=list(range(len_seqdata - end)))   # 開始・終了まで増やすための空行
        seq = cifdata.getsequence(uniprotids)            # PDBファイル内のアミノ酸配列
#        print(seq)
        seq = pd.concat([df_beg, seq, df_end])      # UniProt配列と長さを合わせるため空行を追加
        seq.reset_index(inplace=True, drop=True)         # indexをリセット
        seqdata = pd.concat([seqdata, seq], axis=1)      # UniProt配列(2cycle目以降はUniProt配列＋前cycleまでのPDB配列）とPDB配列を統合
#        print(seqdata)
#        print(seqdata[10:60])
#        print(seqdata[301:350])
#        print(seqdata[-50:-1])
    all_pdblist = [nor_pdblist, sub_pdblist, chi_pdblist, din_pdblist]    # 多次元配列
    if vervose: print(" Data Preparation Finished: " + str(len(nor_pdblist) + len(sub_pdblist) + len(chi_pdblist) + len(din_pdblist)) + "/" + str(len(pdblist)) + " PDB entries, " + str(len(seqdata.columns)-1) + " chains as " + uniprotid)
    if vervose: print(" (Normal PDB: " + str(len(nor_pdblist)) + ", Substitution PDB: " + str(len(sub_pdblist)) + ", Chimera PDB: " + str(len(chi_pdblist)) + ", DelIns PDB: " + str(len(din_pdblist)) + ")")
    return seqdata, all_pdblist

def run_DSA(uniprotid, seqdata, export, seqtype):
    unidata = UniprotData(uniprotid)
    uniprotids = unidata.get_id()                             # あるUniProt IDのXMLに含まれる（複数の）UniProt ID
    str_ids = str(uniprotids)                                      # あるUniProt IDのXMLに含まれる（複数の）UniProt ID
    fasta = unidata.fasta()
    sequence = convert_three(fasta)
    pdblist = unidata.pdblist(method)
    if negative_pdbid != "":
        # negative_pdbidに含まれる要素をpdblistから削除
        # negative_pdbidに含まれるPDBをpdblistから削除（pdblistにない場合にも対応）
        # negative_pdbidをスペースまたはカンマで分割してリストに変換
        negative_list = re.split(r'[,\s]+', negative_pdbid.strip())
        # negative_listをすべて大文字に変換
        negative_list_upper = [neg.upper() for neg in negative_list]
        # pdblistの要素を大文字にして比較し、negative_listに含まれる要素を削除
        pdblist = [item for item in pdblist if item.upper() not in negative_list_upper]
#        for removepdbid in negative_pdbid.split():
#            pdblist.remove(removepdbid.upper())
    trimsequence = sort_sequence(str_ids, seqdata, seq_ratio)
    trimsequence.to_csv(os.path.join(dirpath, f"trimsequence_{uniprotid}.csv"), index=False)
    trimseqcol = trimsequence.columns.values[1:]
#    print(len(trimseqcol))
    if len(trimseqcol) > chain_threshold - 1:
#        print(trimseqcol)
#        print(len(trimseqcol))
#        print(trimsequence[40:70])
        atomcoord = getcoord(trimsequence)
        distance = getdistance2(atomcoord)
        score = getscore(distance, 0)                         # 指定なし: 母数標準偏差, 0: 母数標準偏差, 1: 標本標準偏差

        # distance DataFrame をCSVファイルに書き出す

        # 残基ペアのインデックスを生成
        residue_pairs = list(combinations(atomcoord.index, 2))

        # 残基番号1, 残基番号2 のリストを作成
        residue_num1_list = [pair[0] + 1 for pair in residue_pairs]  # インデックスは0から始まるため、+1する
        residue_num2_list = [pair[1] + 1 for pair in residue_pairs]  # インデックスは0から始まるため、+1する


        # 新しい DataFrame を作成
        residue_num_df = pd.DataFrame({'residue_num1': residue_num1_list, 'residue_num2': residue_num2_list})

        # 距離データを含む列名を取得
        distance_cols = distance.columns[2:]  # 最初の2列 (id, residue pair) を除外

        # 距離データのみを含む新しい DataFrame を作成
        distance_data_df = distance[distance_cols].copy()  # copy() を使用して、元の DataFrame に影響を与えないようにする

        # 2つのDataFrameを結合
        merged_df = pd.concat([residue_num_df, distance_data_df], axis=1)

        merged_df.to_csv(os.path.join(dirpath, f"distance_{uniprotid}.csv"), index=False, header=False)


#  すでに計算済みの平均距離から 閾値以下の cisペアをリストアップし、そのペアについてdistanceリストから 情報を抽出
#        cis_score = score[score['distance mean']<=cis_threshold]
#        if not cis_score.empty:
#            print(cis_score)
#        cis_pair = cis_score.iloc[:, 0].to_list()
#        print(cis_pair)
#        for res_pair in cis_pair:
#            print(res_pair)
#            print(distance[distance.iloc[:, 0] == res_pair])
#  すべての distance情報から 閾値以下の ペアを検索しリストアップ
        cis_index = []
        cis_info = []
        for col in distance.columns.values.tolist()[2:]:    # 列名（PDB Chainの情報のみ）を取得
            tmp = distance.query(f'`{col}`<=@cis_threshold').index.to_list()   # 条件を満たす行のindexをリスト化
            cis_index.extend(tmp)                                              # cis_indexに追記
        if not cis_index:
            cis_info = [[0,0,0,0,0]]
            cis_dist = pd.DataFrame()
        else:
            cis_index = sorted(set(cis_index))                             # 重複除去
            cis_dist = distance.iloc[cis_index, :]                         # distanceから上記のindexのものだけを抽出
#            print(cis_dist)
        # 条件を満たす要素の数をカウント
            cis_cnt = cis_dist.iloc[:,2:].apply(lambda row: (row <= cis_threshold).sum(), axis=1)
            trans_cnt = cis_dist.iloc[:,2:].apply(lambda row: (row > cis_threshold).sum(), axis=1)
            cnt = pd.DataFrame({'cis_cnt': cis_cnt, 'trans_cnt': trans_cnt})

            # trans_cnt が 0 の行のみを抽出
            all_cis_dist = cis_dist[(cnt['trans_cnt'] == 0)]

            mix = ((cnt['cis_cnt'] >= 1) & (cnt['trans_cnt'] >= 1)).sum()
            cis_score = getscore_cis(cis_dist, 0)                          # mean, std, scoreを計算し cis_score へ
            cis_dist = pd.concat([cis_dist, cis_score.iloc[:, 2:]], axis=1)  # cis_distとcis_scoreを連結
            cis_dist = pd.concat([cis_dist, cnt], axis=1)                    # さらにcntを連結
            cis_dist_mean = cis_dist['distance mean'].mean()
            if len(cis_dist['distance mean']) == 1:
                cis_dist_std = 0.00
            else:
                cis_dist_std = cis_dist['distance mean'].std()
            cis_score_mean = cis_dist['score'].mean()
            #cis_num = len(cis_dist)
            cis_num = len(all_cis_dist)
            cis_info.append([cis_dist_mean, cis_dist_std, cis_score_mean, cis_num, mix])
        """
        解析結果の出力
        """
#        print(cis_info)
        log = generate_log_content(unidata.pdbdata, len(sequence), trimsequence, score, cis_info)  # unidata.pdbdataはUniProtデータのPDBに関する部分
        log_output = log.to_string(index=False)

        """
        解析結果の保存
        """
        def export_to_csv(uniprotid, seq_ratio, outputdataname, outputdata, seqtype):
#                dirpath = "drive/MyDrive/Colab Notebooks/Okadaken/Normal-Sub_3/"
#                if not(os.path.exists(dirpath)):
#                    os.makedirs(dirpath)
            filepath = f"{dirpath}{uniprotid}_{str(seq_ratio)}_{outputdataname}_{seqtype}.csv"
            outputdata.to_csv(filepath, index=False)
            if (outputdataname == 'log'):
                unidata.pdbdata.to_csv(filepath, mode='a', index=False)
                trimsequence.to_csv(filepath, mode='a')

        if export == True:
#            export_to_csv(uniprotid, seq_ratio, "log", log, seqtype)
#            export_to_csv(uniprotid, seq_ratio, "score", score.round(4), seqtype)
#            export_to_csv(uniprotid, seq_ratio, "seqdata", seqdata, seqtype)
#            export_to_csv(uniprotid, seq_ratio, "distance", distance, seqtype)
            export_to_csv(uniprotid, seq_ratio, "cis", cis_dist, seqtype)

        return(score, log_output)

    else:
        log = '''

Less than 3 chains'''
        df_blank = pd.DataFrame()
        return(df_blank, log)


In [4]:
#@title 解析
#@title 関数

!pip install biopython
!pip install numba

import requests

from lxml import etree

class UniprotData:
    def __init__(self, uniprot_id: str):
        url = f"https://www.uniprot.org/uniprot/{uniprot_id}.xml"
        self.xml = etree.fromstring(requests.get(url).content)
        self.nsmap = self.xml.nsmap
        TF = self.xml.find('./', self.nsmap).text
        if TF != '\n  ':
            raise KeyError(TF)

    def get_pdb_entries(self):
        """PDBエントリを取得"""
        pdb_entries = self.xml.findall(".//{http://uniprot.org/uniprot}dbReference[@type='PDB']", self.nsmap)
        return pdb_entries

    def get_id(self):
        """UniProt ID取得"""
        return [accession.text for accession in self.xml.findall('./entry/accession', self.nsmap)]

    def fasta(self):
        """FASTA配列の取得"""
        return self.xml.find('./entry/sequence', self.nsmap).text

    def get_fullname(self):
        """UniProtのフルネームを取得"""
        fullname = self.xml.find('./entry/protein/*/fullName', self.nsmap)
        return fullname.text if fullname is not None else "No full name found"

    def get_organism(self):
        """UniProtのオーガニズムを取得"""
        organism = self.xml.find('./entry/organism/name[@type="scientific"]', self.nsmap)
        return organism.text if organism is not None else "No organism found"

    def getpdbdata(self, method):
        """PDBID, method, resolutionの取得"""
        if isinstance(method, str):
            methods = {m.strip() for m in re.split(r'[,\s]+', method) if m.strip()}
        else:
            methods = set(method)

        if not methods:
            methods = {"X-ray", "NMR", "EM"}

        pdbid = []
        data = []
        for dbReference in self.xml.findall('./entry/dbReference[@type="PDB"]', self.nsmap):
            x = []
            for propertys in dbReference:
                value = propertys.attrib["value"]
                x.append(value)
                if value == 'NMR':
                    x.append(None)

            if x and (x[0] in methods):
                pdbid.append(dbReference.attrib["id"])
                data.append(x)

        self.pdbdata = pd.DataFrame(
            data,
            index=pdbid,
            columns=['method', 'resolution', 'position']
        ).T
        return self.pdbdata

    def pdblist(self, method=""):
        """PDBid取得"""
        try:
            return self.pdbdata.columns.tolist()
        except AttributeError:
            return self.getpdbdata(method).columns.tolist()

    def position(self, pdbid):
        """positionの取得"""
        positiondata = self.pdbdata.at["position", pdbid].split(", ")
        if len(positiondata) == 1:
            _, posi = positiondata[0].split("=")
            beg, end = posi.split("-")
            beg = int(beg)
            end = int(end)
        else:
            beg = []
            end = []
            for position in positiondata:
                _, posi = position.split("=")
                align_beg, align_end = posi.split("-")
                beg.append(int(align_beg))
                end.append(int(align_end))
            beg = min(beg)
            end = max(end)
        return beg, end

import os  # osモジュールをインポート
# 出力ディレクトリ設定
dirpath = "drive/MyDrive/Colab Notebooks/Okadaken/Cis_1/"
if not(os.path.exists(dirpath)):
    os.makedirs(dirpath)
import csv  # csvモジュールをインポート
import pytz  # pytzモジュールをインポート
import datetime  # datetimeモジュールをインポート
import shutil  # shutilモジュールをインポート
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from mimetypes import guess_type
import gzip

# 必要な関数をインポート（ドキュメント1と3から）
# ここに全ての関数定義が必要です：
#@title 解析
import requests
from lxml import etree

# UniprotDataクラスは削除（セル1で定義済み）
# 120-129行目のコメントも削除

# 直接130行目から開始
import os
dirpath = "drive/MyDrive/Colab Notebooks/Okadaken/Cis_1/"
# ... 以降のコード ...
# - downloadpdb, convert_three, CifData
# - trim_sequence, trim2_sequence, sort_sequence, getcoord, getdistance2
# - getscore, getscore_cis, generate_log_content
# - count_pdb, prep, run_DSA

# これらの関数をドキュメント1（関数）とドキュメント3（DSAコード）から
# すべてコピーして、この位置に貼り付けてください

# 出力ファイル設定

#@markdown　**条件を設定し実行**<br>
#@markdown UniProt IDを入力（複数の場合は , または スペース 区切り）
UniProtIDs = "P01308" #@param {type:"string"}
# B0Y2Y2,G1UBC6,Q9UN81,B0Y2Y2,G1UBC6,P48825
# O60341(insert), O43451(domainA,domainB)
# Q9UN81,B0Y2Y2(20%),G1UBC6(PDB-0),P48825
# P48825(cis5),Q9F4D5(cis3),Q5JFM9
# P42212,Q5JFM9(X),O32164,Q9YIC2(X),P31151,Q9L387(hetero)
# A0R5M8,A1E280,A2QHE5,A2RJT9,A9JQL9,B2DFG5,B3EY95,B7JBP8,B8NJH3,C4M1P9,D2PPM8,D4Z2G1,E1XUJ2,E3VWI3,K5BDL0,K7WDL7,O00625,O06961,O14684,O14744,O15305,O30298,O32080,O32164,O32323,O34453,O34714,O43924,O49686,O50580,O54288,O57947,O59413,O60760,O64411,O66496,O67082,O70348,O74237,O75531,O75874,O76242,O76745,O80992,O94760,P00004,P00044,P00138,P00163,P00183,P00214,P00268,P00322,P00323,P00327,P00352,P00362,P00371,P00374,P00432,P00437,P00438,P00441,P00442,P00445,P00448,P00452,P00459,P00469,P00489,P00491,P00492,P00509,P00509,P00517,P00586,P00634,P00642,P00720,P00720,P00722,P00760,P00761,P00763,P00772,P00805,P00807,P00811,P00875,P00883,P00915,P00918,P00921,P00924,P00929,P00942,P00953,P01051,P01837,P01958,P02185,P02189,P02213,P02554,P02689,P02787,P02791,P02792,P02794,P02931,P03612,P03958,P04062,P04075,P04117,P04181,P04390,P04395,P04746,P04789,P04825,P04905,P05042,P05050,P05089,P05091,P05093,P05102,P05161,P05310,P05311,P05326,P05413,P05798,P05979,P06132,P06229,P06280,P06628,P06672,P06746,P06766,P06956,P07148,P07328,P07329,P07355,P07379,P07445,P07741,P07798,P07813,P07824,P07896,P07954,P08200,P08263,P08473,P08515,P08536,P08877,P09152,P09186,P09211,P09382,P09455,P09488,P09622,P09936,P09960,P0A017,P0A111,P0A2D5,P0A2K1,P0A6D3,P0A6F5,P0A6I6,P0A6K1,P0A6L2,P0A6L4,P0A6U8,P0A786,P0A7A9,P0A7D4,P0A7F3,P0A7Y4,P0A884,P0A8U6,P0A8V2,P0A953,P0A988,P0A9B2,P0AA04,P0AA25,P0AAI5,P0AB87,P0ABD3,P0ABP8,P0ABQ4,P0ABQ4,P0ABT2,P0ACD8,P0ACE0,P0ACJ8,P0ACP7,P0AD64,P0ADY7,P0AE18,P0AE67,P0AEE5,P0AEK4,P0AEX9,P0AGD3,P0AGE9,P0C0Y7,P0C0Y8,P0C1A9,P0C1Z0,P0C960,P0DP23,P0DP29,P0DUB6,P10340,P10584,P10599,P10760,P11064,P11086,P11310,P11413,P11444,P11558,P11797,P11974,P12004,P12070,P12295,P12676,P12735,P12758,P12807,P12851,P12955,P13123,P13254,P13448,P13479,P13491,P13513,P14174,P14385,P14489,P14618,P14668,P15090,P15494,P15531,P15559,P15587,P15873,P16083,P16113,P16184,P16525,P16544,P16932,P17109,P17612,P18314,P18670,P18886,P19080,P19367,P19573,P19784,P19938,P20279,P20371,P20581,P20586,P20906,P21179,P21673,P21816,P21852,P22069,P22259,P22498,P22643,P22887,P23141,P23295,P23360,P23526,P23657,P23687,P24297,P24300,P24666,P24941,P25910,P26276,P26281,P26935,P27000,P27338,P27487,P28012,P28147,P28161,P28523,P28720,P29082,P29166,P29373,P29401,P29600,P29736,P30014,P30046,P30967,P30986,P31013,P31133,P31151,P31153,P31224,P31947,P32021,P32055,P32396,P34736,P34897,P35270,P35557,P35747,P35755,P36639,P36924,P37019,P38182,P38203,P38489,P38998,P39075,P39116,P39304,P40302,P40859,P40943,P41022,P41365,P42330,P42592,P43379,P44542,P44741,P44859,P45040,P45568,P45718,P45723,P46849,P46881,P46883,P47205,P47811,P47929,P47934,P49356,P49419,P49721,P50120,P50135,P50286,P51668,P51857,P52704,P52895,P54512,P54619,P56119,P56216,P56680,P58687,P59071,P60045,Q03048,Q03243,Q04416,Q04609,Q04760,Q04828,Q05098,Q05315,Q05769,Q06121,Q06GJ0,Q08636,Q08638,Q08751,Q10714,Q12737,Q13126,Q15382,Q15843,Q16539,Q16773,Q16873,Q1EMV2,Q1LCS4,Q24117,Q24451,Q27686,Q27793,Q2FIA5,Q2G506,Q2RSB2,Q31KC7,Q31KC7,Q3JNW6,Q40577,Q44244,Q44467,Q46822,Q47155,Q47PU3,Q4JA33,Q4WAW9,Q4WLV6,Q4WQS0,Q51658,Q54727,Q55891,Q59931,Q973C7,Q97VT7,Q97W02,Q97ZE3,Q980A5,Q99497,Q99685,Q9AIU0,Q9BPX1,Q9BSB4,Q9BY41,Q9YBL2,Q9YBQ2,Q9YIC2,Q9ZMY2,V5YM14
# P10584 P11797 要チェック
#@markdown 構造決定手法を選択
# === 構造決定法のチェックボックス UI ===
use_xray = True   # @param {type:"boolean"}
use_nmr  = False  # @param {type:"boolean"}
use_em   = False   # @param {type:"boolean"}

# 選択結果を集合にまとめる（この集合を以後の関数に渡します）
_selected_methods = []
if use_xray: _selected_methods.append("X-ray")
if use_nmr:  _selected_methods.append("NMR")
if use_em:   _selected_methods.append("EM")

# ひとつも選ばれていない場合は全種類を対象（お好みで挙動変更可）
if not _selected_methods:
    _selected_methods = ["X-ray", "NMR", "EM"]

METHODS_SELECTED = set(_selected_methods)  # 例: {"X-ray","EM"}

#@markdown 解析に使用する配列長の割合(%)
seq_ratio = 20 #@param {type:"number"}
#@markdown　解析に使わないpdb id（複数の場合は スペース 区切り）
negative_pdbid = "" #@param {type:"string"}
#negative_pdbid = "1aaa 1bbs 1BIL 1BIM 1HRN 1RNE 2BKS 2BKT 2FS4 2G1N 2G1O 2G1R 2G1S 2G1Y 2G20 2G21 2G22 2G24 2G26 2G27 3O9L 3OAD 3OAG 3OOT 3OQF 3OQK 3OWN 3Q3T 3Q4B 3Q5H 2IKO 2IKU 2IL2 2REN 2V0Z 2V10 2V11 2V12 2V13 2V16 2X0B 3D91 3G6Z 3G70 3G72 3GW5 3K1W 3KM4 3SFC 3VSW 3VSX 3VUC 3VYD 3VYE 3VYF 4AMT 4GJ5 4GJ6 4GJ7 4GJ8 4GJ9 4GJA 4GJB 4GJC 4GJD 4PYV 2I4Q 4Q1N 4RYC 4RYG 4XX4 5KOQ 5KOS 5KOT 5SXN 5SY2 5SY3 5SZ9 5T4S 5TMG 5TMK"
#negative_pdbid = "1BBS 1BIL 1BIM 1HRN 1RNE 2BKS 2BKT 2FS4 2G1N 2G1O 2G1R 2G1S 2G1Y 2G20 2G21 2G22 2G24 2G26 2G27 3O9L 3OAD 3OAG 3OOT 3OQF 3OQK 3OWN 3Q3T 3Q4B 3Q5H 2IKO 2IKU 2IL2 2REN 2V0Z 2V10 2V11 2V12 2V13 2V16 2X0B 3D91 3G6Z 3G70 3G72 3GW5 3K1W 3KM4 3SFC 3VSW 3VSX 3VUC 3VYD 3VYE 3VYF 4AMT 4GJ5 4GJ6 4GJ7 4GJ8 4GJ9 4GJA 4GJB 4GJC 4GJD 4PYV 2I4Q 4Q1N 4RYC 4RYG 4RZ1 4S1G 4XX3 4XX4 5KOQ 5KOS 5KOT 5SXN 5SY2 5SY3 5SZ9 5T4S 5TMG 5TMK"
#negative_pdbid = "3EVP 3EVR 3EVV 3O77 3O78 3OSQ 3OSR 3U8P 3WLC 3WLD 4IK1 4IK3 4IK4 4IK5 4IK8 4IK9 5F9G 6B7T 6GEL 6GEZ 6UN5 8OVN 8OVO 8OVP 8SMU" #@param {type:"string"}
#negative_pdbid = "4DCX 4I9D 5ON1"
#negative_pdbid = "4DCX 4I9D 5ON1 7A0C"
##@markdown 変異体を解析に含める場合はチェック
#normal = True #@param {type:"boolean"}
#substitution = False #@param {type:"boolean"}
#deletion_insertion = False #@param {type:"boolean"}
#chimera = False #@param {type:"boolean"}
#@markdown CSVファイルに出力する場合はチェック
export = True #@param {type: "boolean"}
#@markdown ヒートマップ（アミノ酸残基 vs Score）を描く場合はチェック
heatmap = True #@param {type: "boolean"}
#@markdown 画面に処理状況を表示する場合はチェック
vervose = True #@param {type: "boolean"}
#@markdown cis解析を行う場合はチェック
proc_cis = True #@param {type:"boolean"}
#@markdown cis結合とみなすCA-CA距離
cis_threshold = 3.3 #@param {type:"number"}
#@markdown データを上書きする
overwrite = True #@param {type:"boolean"}

# 閾値設定
pdb_threshold = 1
chain_threshold = 3

"""
出力ディレクトリ設定
"""
dirpath = "drive/MyDrive/Colab Notebooks/Okadaken/Cis_1/"
if not(os.path.exists(dirpath)):
    os.makedirs(dirpath)
"""
出力ファイル設定
"""
errfilename = f"{dirpath}error.txt"
filename = f"{dirpath}summary.csv"
#if os.path.exists(sumfilepath):
#    sum = pd.read_csv(sumfilepath)
#else:
#    sum = pd.DataFrame(columns=['uniprotid', 'seq_ratio', 'fullName', 'organism', 'Entries', 'Chains', 'Length', 'Length(%)', 'Resolution', 'UMF', 'cis/Length(%)', 'mean_cisDist', 'std_cisDist', 'mean_cisScore', 'cis', 'mix'])
#    sum.to_csv(sumfilepath, index=False)
fieldnames = ['uniprotid', 'seq_ratio', 'fullName', 'organism', 'Entries', 'Chains',
              'Length', 'Length(%)', 'Resolution', 'UMF', 'cis/Length(%)', 'mean_cisDist', 'std_cisDist', 'mean_cisScore', 'cis', 'mix']

# === 既存 summary.csv の読み込み＆バックアップ ===
fieldnames = [
    'uniprotid','seq_ratio','fullName','organism',
    'Entries','Chains','Length','Length(%)','Resolution','UMF',
    'cis/Length(%)','mean_cisDist','std_cisDist','mean_cisScore','cis','mix'
]

existing_data = []
if os.path.exists(filename):
    jst = pytz.timezone('Asia/Tokyo')
    timestamp = datetime.datetime.now(jst).strftime('%Y%m%d_%H%M%S')
    backup_filename = f"{dirpath}summary_backup_{timestamp}.csv"
    shutil.copy2(filename, backup_filename)
    print(f'Backup created: {backup_filename}')
    with open(filename, 'r') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            existing_data.append(row)
else:
    # ファイルがなければヘッダだけ作成
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

# === ここから解析ループ ===
ids = [x.strip() for x in re.split(r'[,\s]+', UniProtIDs.strip())]

for i, uniprotid in enumerate(ids):
    try:
        if vervose:
            print("#####################################################################################")
            print("Processing " + uniprotid + " ...")
        unidata = UniprotData(uniprotid)
    except Exception as e:
        print(f"Error processing {uniprotid}: {e}\n")
        with open(errfilename, 'a') as err:
            err.write(f"Error processing {uniprotid}: {e}\n")
        continue

    fullName = unidata.get_fullname()
    organism = unidata.get_organism()
    if vervose:
        print(fullName + ' from ' + organism)
        print("### Preparation #########################################")

    # ここもループ内に置く
    pngfilepath = f"{dirpath}{uniprotid}_{str(seq_ratio)}_heatmap.png"
    txtfilepath = f"{dirpath}{uniprotid}_{str(seq_ratio)}_summary.txt"

    # ✅ X-ray/EM のみでカウントして 3 未満ならスキップ
    if not count_pdb(uniprotid, methods=METHODS_SELECTED):
        print("Less than 3 PDB entries (within selected methods)")
        if vervose:
            print("###############################################")
        continue  # ← ここはループ内なのでOK

    # ✅ 通過した場合だけ準備を実行（if の外）
    seqdata, all_pdblist = prep(uniprotid, methods=METHODS_SELECTED)
    seqdata1 = seqdata.filter(like=uniprotid)

    # …以降、run_DSA など続き




    # normal+sub のみを使う（必要なら normal / sub も同様に呼ぶ）
    normal = True
    substitution = True
    seqtype = 'nor+sub'
    pdbtuple = tuple(all_pdblist[0] + all_pdblist[1])
    if vervose:
        print("")
        print("### normal & mutant #####################################")
        print("PDB: ", pdbtuple)
        print(len(pdbtuple), "entries were processed")

    seqdata2 = seqdata.loc[:, seqdata.columns.str.startswith(pdbtuple)]
    if vervose:
        print(seqdata2.shape[1], "chains are being processed ...")

    # 入力データ整形
    norsub_seqdata = pd.concat([seqdata1, seqdata2], axis=1)

    # --- ここで run_DSA を呼ぶ（必ず run_DSA が df_all を第3戻り値で返すこと）---
    sc_all, log_all, df_all = run_DSA(uniprotid, norsub_seqdata, export, seqtype, methods=METHODS_SELECTED)

    # df_all をそのまま使用。log_all のパースはやめる
    if df_all is None or len(df_all)==0:
        print(f"Error: df_all is empty for {uniprotid}. Skipped.")
        continue

    # 必要列がそろっているかチェック
    required_cols = {
        'Entries','Chains','Length','Length(%)','Resolution','UMF',
        'cis/Length(%)','mean_cisDist','std_cisDist','mean_cisScore','cis','mix'
    }
    if not required_cols.issubset(set(df_all.columns)):
        print(f"Error: df_all columns missing for {uniprotid}. Got columns: {list(df_all.columns)}")
        continue

    # 1行目を使って new_entry を作る
    row0 = df_all.iloc[0]
    new_entry = {
        'uniprotid': uniprotid,
        'seq_ratio': float(seq_ratio),
        'fullName': fullName,
        'organism': organism,
        'Entries': int(row0['Entries']),
        'Chains': int(row0['Chains']),
        'Length': int(row0['Length']),
        'Length(%)': float(row0['Length(%)']),
        'Resolution': float(row0['Resolution']),
        'UMF': float(row0['UMF']),
        'cis/Length(%)': float(row0['cis/Length(%)']),
        'mean_cisDist': float(row0['mean_cisDist']),
        'std_cisDist': float(row0['std_cisDist']),
        'mean_cisScore': float(row0['mean_cisScore']),
        'cis': int(row0['cis']),
        'mix': int(row0['mix']),
    }

    # テキストの概要出力（必要に応じて）
    with open(txtfilepath, mode='w') as f:
        print("### " + uniprotid + " (seq_ratio= " + str(seq_ratio) + ") Summary ######################", file=f)
        print(fullName, file=f)
        print(organism, file=f)
        # df_all の表を追記
        print(df_all.to_string(index=False), file=f)

    # まとめ CSV の更新（同じ uniprotid & seq_ratio があれば上書き）
    found = False
    for i, existing_entry in enumerate(existing_data):
        if (existing_entry['uniprotid'] == new_entry['uniprotid'] and
            float(existing_entry['seq_ratio']) == new_entry['seq_ratio']):
            if overwrite:
                existing_data[i] = {k: str(v) for k, v in new_entry.items()}  # 文字列で保存
            found = True
            break
    if not found:
        existing_data.append({k: str(v) for k, v in new_entry.items()})

    # ここに貼る（インデントは上と同じレベル）
    if heatmap:
        # sc_all を使って描画（必要ならあなたの既存コードを貼る）
        pass


    # 可視化（必要なら）
    if heatmap:
        # sc_all からヒートマップ生成（あなたの既存の関数を使用）
        pass  # ここは今まで通りでもOK

    if vervose:
        print("Processing " + uniprotid + " Finished")
        print("#####################################################################################")
        print("")

# === すべての ID が終わったら一度だけ CSV 書き出し ===
with open(filename, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in existing_data:
        writer.writerow(row)

print(f"Update '{filename}'")
if vervose: print("Job Completed")





Backup created: drive/MyDrive/Colab Notebooks/Okadaken/Cis_1/summary_backup_20251026_160235.csv
#####################################################################################
Processing P01308 ...
Insulin from Homo sapiens
### Preparation #########################################


TypeError: count_pdb() got an unexpected keyword argument 'methods'